In [ ]:
# ============================================================================
# 1. IMPORTING THE NECESSARY LIBRARIES
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from scipy.integrate import solve_ivp
import os

if not os.path.exists('figures'):
    os.makedirs('figures')

# =============================================================================
# 2. PARAMETER INITIALIZATION & DATA
# =============================================================================
# The population and rate constants based on WHO clinical standards for P. falciparum
N = 1_000_000.0  
gamma_base = 1/7.0  # Recovery rate(Mean infectious period of 7 days) 
sigma = 1/12.0      # Incubation rate (Mean hepatic stage of 12 days) 

# Surveillance data from the observed January surge (Northern Namibia)
observed_weeks = np.array([1, 2, 3, 4])
observed_cases = np.array([1800, 2200, 2450, 2580])
observed_times = observed_weeks * 7

# Initial conditions (S, E, I, R)
# Included an exposed compartment to model the latent parasite reservoir
I0 = 8760.0
E0 = 2.5 * I0  # Heuristic estimation of latent burden before peak      
S0 = N - E0 - I0
y0 = [S0, E0, I0, 0.0]

# =============================================================================
# 3. BAYESIAN INFERENCE
# =============================================================================
# Defining the ODE system using PyTensor for compatibility with HMC samplers
def seir_ode_pymc(y, t, p):
    S, E, I, R = y
    beta_0, eta = p[0], p[1]
    # Seasonal forcing: beta fluctuates to model mosquito density peaks (Rainy season)
    beta_t = beta_0 * (1 + eta * pt.cos(2 * np.pi * (t - 105) / 365))
    return [-beta_t*S*I/N, beta_t*S*I/N - sigma*E, sigma*E - gamma_base*I, gamma_base*I]

# Wrapping the ODE solver as a PyMC probability component
from pymc.ode import DifferentialEquation
seir_model_wrapper = DifferentialEquation(func=seir_ode_pymc, times=observed_times, n_states=4, n_theta=2, t0=0)

with pm.Model() as malaria_model:
    # Incorporating clinical knowledge into parameter bounds
    beta_0 = pm.Lognormal("beta_0", mu=np.log(0.4), sigma=0.3) # Mean transmission intensity
    eta = pm.Beta("eta", alpha=2, beta=5) # Seasonal amplitude (0 to 1)
    phi = pm.HalfNormal("phi", sigma=10) # Overdispersion parameter

    # Mapping ODE output to observed symptomatic cases
    sol = seir_model_wrapper(y0=y0, theta=[beta_0, eta])
    # Negative Binomial to account for noise and overdispersion in hospital data
    pm.NegativeBinomial("Y_obs", mu=sol[:, 2], alpha=phi, observed=observed_cases)
    
    # Using the No-U-Turn Sampler (NUTS) for high dimensional convergence
    trace = pm.sample(1000, tune=1000, target_accept=0.9, chains=2)
    pm.sample_posterior_predictive(trace, extend_inferencedata=True)

# Extracting point estimates for deterministic validation
post_beta = trace.posterior["beta_0"].mean().item()
post_eta = trace.posterior["eta"].mean().item()

# =============================================================================
# 4. VISUALIZATIONS
# =============================================================================
plt.style.use('seaborn-v0_8-whitegrid')
t_eval = np.linspace(0, 210, 210)

def seir_scipy(t, y, b0, e, g):
    bt = b0 * (1 + e * np.cos(2 * np.pi * (t - 105) / 365))
    return [-bt*y[0]*y[2]/N, bt*y[0]*y[2]/N - sigma*y[1], sigma*y[1] - g*y[2], g*y[2]]

res = solve_ivp(seir_scipy, [0, 210], y0, args=(post_beta, post_eta, gamma_base), t_eval=t_eval)

# FIG 1: Mass Action Balance
# Demonstrates how the population transitions through disease states over time
plt.figure(figsize=(8, 5))
plt.stackplot(t_eval, res.y[2], res.y[1], res.y[0], labels=['I', 'E', 'S'], colors=['#e74c3c', '#f39c12', '#3498db'], alpha=0.7)
plt.title("FIG 1: Full SEIR State Transitions", fontweight='bold')
plt.xlabel("Days"), plt.ylabel("Individuals"), plt.legend()
plt.savefig('figures/fig1_seir_states.png', dpi=300, bbox_inches='tight')

# FIG 2: Uncertainty Quantification
# The Highest Density Interval (HDI) proves the model captures the data within statistical bounds
plt.figure(figsize=(8, 5))
az.plot_hdi(observed_times, trace.posterior_predictive["Y_obs"], color="gray", fill_kwargs={"alpha": 0.3, "label": "94% HDI"})
plt.plot(t_eval, res.y[2], color='black', label="Posterior Mean")
plt.scatter(observed_times, observed_cases, color='red', label="Observed Data", zorder=5)
plt.title("FIG 2: Model Calibration & Uncertainty", fontweight='bold')
plt.legend()
plt.savefig('figures/fig2_calibration.png', dpi=300, bbox_inches='tight')

# FIG 3: Reproduction Number Dynamics
# Captures the seasonal oscillation of R(t), the key metric for outbreak control
plt.figure(figsize=(8, 5))
Rt = (post_beta * (1 + post_eta * np.cos(2*np.pi*(t_eval-105)/365))) / gamma_base
plt.plot(t_eval, Rt, color='#8e44ad', linewidth=2)
plt.axhline(1, color='black', linestyle='--')
plt.title("FIG 3: Seasonal Effective Reproduction Number R(t)", fontweight='bold')
plt.ylabel("R(t)"), plt.xlabel("Days")
plt.savefig('figures/fig3_rt_dynamics.png', dpi=300, bbox_inches='tight')

# FIG 4: Phase Space Portrait
# Illustrates the geometric stability and S-I relationship of the dynamical system
plt.figure(figsize=(8, 5))
plt.plot(res.y[0], res.y[2], color='#2c3e50', lw=2)
plt.title("FIG 4: Epidemiological Phase Portrait", fontweight='bold')
plt.xlabel("Susceptible (S)"), plt.ylabel("Infectious (I)")
plt.savefig('figures/fig4_phase_portrait.png', dpi=300, bbox_inches='tight')

## FIG 5: Sensitivity Analysis 
# Explores the parameter space to identify which factors lead to the highest clinical load
plt.figure(figsize=(8, 5))
b_range, g_range = np.linspace(0.2, 0.6, 15), np.linspace(0.1, 0.2, 15)
peaks = np.array([[np.max(solve_ivp(seir_scipy, [0, 210], y0, args=(b, post_eta, g), t_eval=t_eval).y[2]) for g in g_range] for b in b_range])
plt.imshow(peaks, origin='lower', extent=[0.1, 0.2, 0.2, 0.6], aspect='auto', cmap='YlOrRd')
plt.colorbar(label="Peak Infection Count")
plt.title("FIG 5: Strategic Sensitivity Analysis", fontweight='bold')
plt.xlabel("Recovery Rate (Gamma)"), plt.ylabel("Transmission Rate (Beta)")
plt.savefig('figures/fig5_sensitivity.png', dpi=300, bbox_inches='tight')

# FIG 6: Policy Simulation 
plt.figure(figsize=(8, 5))
res_nets = solve_ivp(seir_scipy, [0, 210], y0, args=(0.7 * post_beta, post_eta, gamma_base), t_eval=t_eval)
plt.fill_between(t_eval, res.y[2], res_nets.y[2], color='green', alpha=0.2, label="Averted Cases")
plt.plot(t_eval, res.y[2], 'k--', alpha=0.4, label="Baseline")
plt.plot(t_eval, res_nets.y[2], color='green', label="Bed-Net Strategy (30% reduction)")
plt.title("FIG 6: Mitigation Strategy Impact", fontweight='bold')
plt.legend()
plt.savefig('figures/fig6_intervention.png', dpi=300, bbox_inches='tight')

/opt/anaconda3/lib/python3.13/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta_0, eta, phi]


Output()